# Communicating robots

This notebook illustrates the communication abstractions of the simulator, independently of any particular paper (see DESIGN-COMMUNICATION.md). There are three parts:

1. **The medium on its own** (`communication.py`):
   - robots register under unique names;
   - a message is broadcast or sent to one robot;
   - every recipient gets its own copy;
   - messages are stamped with `(timestep, round)`;
   - only the robot that owns a mailbox can read it.
2. **Communication rounds in a simulation.** At the start of every timestep, `simulate_1day` runs `communication_rounds` rounds. In each round, every robot's `policy.act_send(round)` runs first, then every robot's `policy.act_receive(round, messages)`. Every `Policy` has no-op versions of both, so a policy communicates by overriding them.
3. **A different communication model.** Papers with other models, such as limited range, loss or delay, subclass the medium. The notebook defines a range-limited medium and shows how the number of rounds then matters, because information must be relayed.

In [ ]:
import math
import sys
import tempfile
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from communication import Message, PerfectCommunicationMedium
from policy import FollowPathPolicy
from robot import Robot
from water_berry_farm import MiniberryFarm, WaterberryFarmEnvironment, WBF_IM_DiskEstimator, WBF_Score_WeightedAsymmetric
from wbf_simulate import simulate_1day

farm = MiniberryFarm(scale=1)
farm.create_type_map()
environment = WaterberryFarmEnvironment(farm, use_saved=False, seed=10, savedir=tempfile.mkdtemp())
environment.proceed(6)

## 1. The medium on its own

`add_robot` registers a robot under its name, creates its mailbox, and sets `robot.com`, through which its policy sends. `send(sender, destination, message)` takes a robot name, or `None` for a broadcast to everyone except the sender. `receive(robot)` drains the robot's mailbox.

By convention, a message's content is a dict with a `"type"` key and plain data values.

In [ ]:
medium = PerfectCommunicationMedium(environment)
alpha, beta, gamma = Robot("alpha", 0, 0, 0), Robot("beta", 0, 0, 0), Robot("gamma", 0, 0, 0)
for robot in [alpha, beta, gamma]:
    medium.add_robot(robot)
print("alpha.com is the medium:", alpha.com is medium)

# simulate_1day sets the current (timestep, round) before every round; here we set it by hand
medium.timestep, medium.round = 3, 0
alpha.com.send(alpha, None, Message({"type": "hello", "position": [0, 0]}))   # broadcast
alpha.com.send(alpha, "gamma", Message({"type": "private", "text": "only for gamma"}))  # directed

for robot in [alpha, beta, gamma]:
    print(robot.name, "receives", medium.receive(robot))
print("a second receive finds the mailbox empty:", medium.receive(beta))

**Every recipient gets its own deep copy.** The sender may change the content afterwards, and recipients may change theirs, without affecting each other. This is why shared objects must be referred to by ids, never passed by reference.

In [ ]:
content = {"type": "plan", "waypoints": [[1, 1], [2, 2]]}
alpha.com.send(alpha, None, Message(content))
content["waypoints"].append([9, 9])  # changed after sending
at_beta, at_gamma = medium.receive(beta)[0], medium.receive(gamma)[0]
at_beta.content["waypoints"].clear()  # beta changes its copy
print("beta: ", at_beta.content)
print("gamma:", at_gamma.content)

**Only the owner can read a mailbox.** Receiving for another robot object of the same name raises an exception:

In [ ]:
try:
    medium.receive(Robot("beta", 0, 0, 0))
except Exception as exception:
    print("Exception:", exception)

## 2. Communication rounds in a simulation

The example policy follows a path, and shares what it observes. `SharingPolicy` broadcasts every cell it observes, and keeps `known`, its own view of which cells the team has observed and by whom. With `relay=True`, it also forwards news it receives in the next round. This only matters when not everyone hears everyone (Part 3).

The robots' observations arrive after they move, so a cell observed at timestep `t` is broadcast in the rounds of timestep `t+1`.

In [ ]:
class SharingPolicy(FollowPathPolicy):
    """Follows a path and shares the observed cells; known: (x, y) -> name of the observer"""
    def __init__(self, waypoints, relay=True):
        super().__init__(1, waypoints)
        self.relay = relay
        self.known = {}
        self.news = [] # the cells to send in the next round

    def add_observation(self, obs):
        self.known[(obs["x"], obs["y"])] = obs["robot"]
        self.news.append({"x": obs["x"], "y": obs["y"], "observer": obs["robot"]})

    def act_send(self, round):
        for item in self.news:
            self.robot.com.send(self.robot, None, Message({"type": "observed", **item}))
        self.news = []

    def act_receive(self, round, messages):
        for message in messages:
            cell = (message.content["x"], message.content["y"])
            if cell not in self.known:
                self.known[cell] = message.content["observer"]
                if self.relay:
                    self.news.append({"x": cell[0], "y": cell[1], "observer": message.content["observer"]})

A medium is an ordinary class, so instrumenting it is a subclass. `LoggingMedium` records every send with its `(timestep, round)`. The three robots drive up three parallel columns, x = 1, 5 and 9.

In [ ]:
class LoggingMedium(PerfectCommunicationMedium):
    """A perfect medium that records every message sent"""
    def __init__(self, env):
        super().__init__(env)
        self.log = []

    def send(self, sender, destination, message):
        self.log.append((self.timestep, self.round, sender.name, destination, message.content["type"]))
        super().send(sender, destination, message)

paths = {"alpha": [[1, 0], [1, 10]], "beta": [[5, 0], [5, 10]], "gamma": [[9, 0], [9, 10]]}

def run(medium, communication_rounds, relay=True, timesteps=12):
    """Runs the robots on the columns; returns their policies, and how many of alpha's cells 
    gamma knows at the end of every timestep"""
    robots = []
    for name, path in paths.items():
        robot = Robot(name, path[0][0], path[0][1], 0)
        robot.assign_policy(SharingPolicy(path, relay))
        medium.add_robot(robot)
        robots.append(robot)
    gamma_knows = []
    def after_timestep(results, environment, robots, estimator, evaluator):
        gamma_knows.append(sum(1 for observer in robots[2].policy.known.values() if observer == "alpha"))
    simulate_1day(environment=environment, robots=robots, estimator=WBF_IM_DiskEstimator(farm.width, farm.height),
                  evaluator=WBF_Score_WeightedAsymmetric(), timesteps=timesteps, estimator_interval=4,
                  communication=medium, communication_rounds=communication_rounds, after_timestep=after_timestep)
    return {robot.name: robot.policy for robot in robots}, gamma_knows

medium = LoggingMedium(environment)
policies, _ = run(medium, communication_rounds=1)
print("(timestep, round, sender, destination, type) of the first messages:")
for entry in medium.log[:6]:
    print("   ", entry)

At timestep 2, alpha sends three messages: its own new cell, and relays of the cells beta and gamma announced in timestep 1. Under a perfect medium these relays are redundant, since every robot already heard the originals. They become useful in Part 3.

With a perfect medium and one round, every robot knows every observed cell, with the observer credited. Without communication (`communication_rounds=0`), every robot knows only its own cells:

In [ ]:
policies_silent, _ = run(LoggingMedium(environment), communication_rounds=0)
for name in paths:
    print(f"{name}: knows {len(policies[name].known)} cells with communication, {len(policies_silent[name].known)} without")

## 3. A different communication model: limited range

Another communication model is another subclass. It overrides `send` (and, if needed, `receive`), and keeps the mailbox and stamping behavior of the base class. `RangeLimitedMedium` delivers a message only to the robots within `communication_range` of the sender, at the robots' positions when the round runs.

With a range of 4.5, the neighboring columns hear each other (distance 4), but alpha and gamma (distance 8) do not. Gamma therefore learns about alpha's cells only when beta relays them.

In [ ]:
class RangeLimitedMedium(LoggingMedium):
    """Delivers a message only to the robots within communication_range of the sender"""
    def __init__(self, env, communication_range):
        super().__init__(env)
        self.communication_range = communication_range

    def send(self, sender, destination, message):
        for name, robot in self.robots.items():
            if name == sender.name or (destination is not None and destination != name):
                continue
            if math.dist((sender.x, sender.y), (robot.x, robot.y)) <= self.communication_range:
                super().send(sender, name, message)

runs = {
    "perfect medium, 1 round": run(LoggingMedium(environment), 1),
    "range 4.5, 1 round": run(RangeLimitedMedium(environment, 4.5), 1),
    "range 4.5, 2 rounds": run(RangeLimitedMedium(environment, 4.5), 2),
    "range 4.5, 2 rounds, no relaying": run(RangeLimitedMedium(environment, 4.5), 2, relay=False),
}
for label, (_, gamma_knows) in runs.items():
    print(f"{label:34s} gamma knows of alpha's cells: {gamma_knows}")

- **Perfect medium:** gamma hears alpha directly, one timestep after each observation.
- **Range, 1 round:** beta receives alpha's news in round 0, but can forward it only in the next round, which is the next timestep. Gamma is one timestep further behind.
- **Range, 2 rounds:** beta forwards in round 1 of the same timestep, so gamma keeps up with the perfect medium. The number of rounds is the number of hops information travels per timestep.
- **No relaying:** gamma never learns about alpha's cells. A limited medium needs policies that cooperate in forwarding.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, (_, gamma_knows) in runs.items():
    if label.startswith("perfect"):  # drawn wide underneath: 2 rounds with range 4.5 coincides with it
        ax.plot(gamma_knows, color="lightgray", linewidth=8, label=label)
    else:
        ax.plot(gamma_knows, marker="o", label=label)
ax.set_xlabel("timestep")
ax.set_ylabel("alpha's cells known to gamma")
ax.legend()
plt.show()

What each robot knows at the end, with range 4.5 and 1 round, colored by the observer. Beta, in range of both, knows everything. Alpha and gamma know each other's columns only through beta's relays, which arrive one timestep late. Each of them is therefore missing the other's last cell (the top of the column): the day ended before it was relayed.

In [ ]:
names = sorted(paths)
colors = ["tab:blue", "tab:orange", "tab:green"]
policies_range = runs["range 4.5, 1 round"][0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, name in zip(axes, names):
    view = np.full((farm.width, farm.height), np.nan)
    for (x, y), observer in policies_range[name].known.items():
        view[x, y] = names.index(observer)
    ax.imshow(view.T, origin="lower", cmap=ListedColormap(colors), vmin=-0.5, vmax=len(names) - 0.5)
    ax.set_title(f"known to {name}")
for name, color in zip(names, colors):
    axes[0].plot([], [], "s", color=color, label=f"observed by {name}")
fig.legend(loc="lower center", ncol=3, fontsize=9)
plt.show()

## 4. Writing a communication model or a communicating policy

- **A medium** subclasses `CommunicationMedium` (or `PerfectCommunicationMedium`) and implements `send(sender, destination, message)` and `receive(receiver)`. The base class provides:
  - registration (`add_robot`), with unique names;
  - the mailboxes;
  - the current `timestep` / `round`, set by `simulate_1day`, for stamping.

  Loss, delay or bandwidth limits are decisions in `send`, or deferred deliveries released in `receive`.
- **A policy** overrides `act_send(round)` and `act_receive(round, messages)`, and sends through `self.robot.com`. In each round, every robot sends before any robot receives. A message sent in round `r` is therefore received in round `r`, and a reply needs another round.
- **Messages** carry plain data, with a `"type"` key. Objects in a message arrive as copies, so they must be identified by ids.
- **The simulation:** pass the medium and `communication_rounds` to `simulate_1day`. Register every robot with the medium before the run.